# Sıfırdan GPT

Sana bir sırrımız var. Muhtemelen sen de zaten biliyorsun ama herkes söylemeye çekiniyor:


Kimse ticket açıklaması yazmayı sevmez 🙄


Bu challenge'da, gerçekçi görünen ticket'lar üreten kendi GPT tarzı modelini geliştireceksin; böylece çalışkan asistanların (TA'ların) için o uzun açıklamaları yazmaya artık zaman harcamayacaksın (burada bir tane üretip direkt yapıştırabilirsin!) 😮‍💨

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/06-DL/smarter_not_harder.jpg" width = "300px">

Bu notebook hem teoriyi hem pratiği kapsıyor. Sadece adımları uygulamak yerine, yaptıklarımızın __neden__ yapıldığını gerçekten anlamaya zaman ayır. 

👉👉👉 Bu günün en büyük challenge'ı; çünkü dersteki temel kavramların hepsini pekiştireceğiz. O yüzden gününün çoğunu alırsa stres yapma. 👈 👈 👈 

Her adımı yavaş ve metodik şekilde ilerleteceğiz; büyük resim kabaca şöyle:

### 1️⃣ Veri Ön İşleme 📊

Ticket verimizi okuyacağız, temizleyeceğiz ve eğitim/test/validasyon TensorFlow dataset'lerine böleceğiz. 🧹🔀

### 2️⃣ Sözlük (Vocabulary) Oluşturma 📚

Başlangıç için TensorFlow'dan basit bir TextVectorization katmanı kullanacağız ve özel bir metin standardizasyon fonksiyonu ile metni token'lara çevireceğiz. Ayrıca token'lar ile orijinal metin arasında rahatça dönüşüm yapabildiğimizden emin olacağız. 🗂️

### 3️⃣ Model Oluşturma ve Eğitim 🔨

Transformer tabanlı bir yaklaşımla metin üretim modelinin mimarisini tanımlayacağız. Asıl ağır işin büyük kısmı burada yapılacak 🏗️🧠

### 4️⃣ Metin Üretimi 📝🔮

Son olarak, her epoch sonunda örnek metin üreten bir callback fonksiyonu yazacağız. Modelinin yaratıcılığı parlasın! ✨🌟


### 🎉5️⃣ 🎉 Serbest Çalışma

Bu noktada NLP konusunda epey iyi olacaksın; bu bölümde yazdığın kodu toparlayıp düzenleyebilir ve kendi dataset'lerinle oynamaya başlayabilirsin. Keşfetmeye ve eğlenmeye hazır ol!

## Bu kadar giriş yeter! Hadi başlayalım!

Birazdan bu kütüphaneye ihtiyacın olacak; şimdilik aşağıdaki hücreyi çalıştırıp `keras_nlp` kütüphanesini yükle:

In [2]:
!pip install keras-nlp

In [3]:
import tensorflow as tf

2026-04-15 11:01:32.769242: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 11:01:32.781228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 11:01:32.794727: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 11:01:32.794782: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 11:01:32.806699: I tensorflow/core/platform/cpu_feature_gua

## 1️⃣ Veri Ön İşleme 📊

Aşağıdaki hücreyi çalıştırarak bu [linkten](https://d37p7d5kaxknzw.cloudfront.net/projects/tickets.txt) `tickets.txt` dosyasını indir ve `data/` klasörüne koy. Sonra txt dosyasını bir değişkene yükle — bu değişken tek, uzun bir string olacak:

In [4]:
!mkdir -p data
!curl https://d37p7d5kaxknzw.cloudfront.net/projects/tickets.txt > data/tickets.txt
with open("data/tickets.txt", "r") as f:
    text = f.read()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  123k  100  123k    0     0   155k      0 --:--:-- --:--:-- --:--:--  155k


String'in ilk 1000 karakterini yazdır.

In [5]:
text[:1000]

"I'm developing a sentiment analysis model for customer reviews, but I'm struggling with handling domain-specific language or sarcasm. What are some techniques like domain adaptation, transfer learning, or using pre-trained language models such as BERT or GPT-3 that can help me improve sentiment analysis performance on such challenging data? --- I'm facing challenges in detecting and handling outliers in my numerical data. How can I use techniques like the interquartile range (IQR), Z-score, or robust statistical methods to identify outliers and decide whether to remove them or treat them differently in my analysis or modeling pipeline? --- I'm working on a collaborative filtering-based recommendation system, and I need guidance on handling cold start problems when dealing with new users or items with limited interaction data. How can I leverage techniques like content-based filtering, popularity-based recommendations, or hybrid approaches to address the cold start issue? --- I'm encou

Her ticket bazı tirelerle ayrılmış. Metni tirelerden böl (split et). 

In [6]:
tickets = text.split("---")
tickets

["I'm developing a sentiment analysis model for customer reviews, but I'm struggling with handling domain-specific language or sarcasm. What are some techniques like domain adaptation, transfer learning, or using pre-trained language models such as BERT or GPT-3 that can help me improve sentiment analysis performance on such challenging data? ",
 " I'm facing challenges in detecting and handling outliers in my numerical data. How can I use techniques like the interquartile range (IQR), Z-score, or robust statistical methods to identify outliers and decide whether to remove them or treat them differently in my analysis or modeling pipeline? ",
 " I'm working on a collaborative filtering-based recommendation system, and I need guidance on handling cold start problems when dealing with new users or items with limited interaction data. How can I leverage techniques like content-based filtering, popularity-based recommendations, or hybrid approaches to address the cold start issue? ",
 " I'

İlk görevimiz, oluşturduğumuz her string'in sonuna " EOS " eklemek. Bu, modele cümlenin sonuna geldiğini söyleyecek.

In [7]:
"""Her ticket string'inin sonuna cümle sonu marker'ı olarak `EOS` ekle (`standardize` sonrası vocabulary'de `eos`)."""
tickets = [t.strip() + " EOS" for t in tickets]


Kaç tane ticket'ın olduğunu ve en uzun ticket açıklamasının (__karakter değil, kelime cinsinden__) uzunluğunu kontrol et. Bu maksimum uzunluğu `max_len` değişkenine kaydet.

In [8]:
"""En uzun ticket'ı kelime (`split`) sayısıyla ölç; `EOS` dahil `max_len` olur."""
max_len = max(len(ticket.split()) for ticket in tickets)


In [9]:
print(f"There are {len(tickets)} tickets in the dataset")
print(f"The longest ticket description is {max_len} words (including the 'EOS' word)")

There are 372 tickets in the dataset
The longest ticket description is 56 words (including the 'EOS' word)


Sıradaki iş, güzelce hazırladığımız cümle koleksiyonunu token'lara çevirmek. Bunun için basit bir [TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) katmanı kullanacağız.

Daha gelişmiş tokenization tekniklerine sonra değineceğiz; şimdilik `tokenize_layer` adında bir tokenizer katmanı oluştur: cümleleri "standardize" ederek küçük harfe çevirsin (şimdilik noktalama işaretlerini dert etmiyoruz), integer çıktılar üretsin ve maksimum cümle uzunluğu `max_len` olsun. Bu adımları nasıl yapacağına dair dokümanlara / docstring'lere bakabilirsin.

In [10]:
from keras.layers import TextVectorization

"""`TextVectorization`: `standardize='lower'`, `output_mode='int'`, sabit `output_sequence_length` (padding/truncation)."""
vectorize_layer = TextVectorization(
    standardize="lower",
    output_mode="int",
    output_sequence_length=max_len,
)


2026-04-15 11:01:38.752922: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 11:01:38.821878: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Katmanı oluşturduktan sonra, cümlelerimize `adapt()` uygulamamız gerekiyor (yani `tickets` değişkenini `vectorize_layer.adapt()` fonksiyonuna ver). Bu katman `adapt` edildiğinde dataset'i analiz eder, her bir string değerinin sıklığını belirler ve bunlardan bir vocabulary oluşturur.

Sonrasında vocabulary'yi [get_vocabulary()](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization#get_vocabulary) metodu ile inceleyebiliriz.

Üretilen listeyi `vocab` adlı bir değişkene ata ve ilk 10 elemana göz at.

In [11]:
"""`adapt`: korpus üzerinde frekans sayıp `vocabulary` üretir."""
vectorize_layer.adapt(tickets)
vocab = vectorize_layer.get_vocabulary()
vocab[:10]


['', '[UNK]', 'or', 'like', 'and', "i'm", 'eos', 'can', 'techniques', 'to']

In [12]:
vocab_size = len(vocab)


In [13]:
from nbresult import ChallengeResult

result = ChallengeResult('vocab',
    vocab_size = vocab_size
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 1 item

test_vocab.py::TestVocab::test_vocab PASSED                              [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/vocab.pickle

git commit -m 'Completed vocab step'

git push origin master



Aşağıdaki hücreyi çalıştırarak token'larını tekrar kelimelere çevirmene yardımcı olacak bir sözlük (dictionary) oluştur:



In [14]:
index_lookup = dict(zip(range(len(vocab)), vocab))

Seçtiğin örnek bir cümle üzerinde vectorizer layer'ı dene (ör. `vectorize_layer("Try me")`). Çıktıda integer'larla dolu bir tensor görmelisin!

In [15]:
"""Örnek cümle -> `(max_len,)` `token` id tensor'u."""
vectorize_layer("Try me")


<tf.Tensor: shape=(56,), dtype=int64, numpy=
array([843,  46,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0])>

Sonra, oluşturduğun sözlükle token'ları tekrar kelimelere çevirebildiğinden emin ol.
<details >
<summary>İpucu için tıkla 👇</summary>
<br>
Vectorizer bir tensor döndürüyor; içinde döngü kurmak için bunu tekrar sayı listesine çevirmen gerekecek. Tensor üzerinde <code>.numpy().tolist()</code> dene.
</details>

In [16]:
"""Id listesini `numpy().tolist()` ile alıp `index_lookup` ile kelimelere çevir."""
ids = vectorize_layer("Try me").numpy().tolist()
[index_lookup[i] for i in ids]


['try',
 'me',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '']

Vocab'umuz oldukça küçük olduğu için, sözlükte olmayan bir kelime girersen bazı `[UNK]` token'ları görebilirsin; endişe etme!

Vectorizer ile rahat ettiğine göre, şimdi tüm cümleleri dolaşıp her birini tokenize edelim! Bu listeyi `all_tokenized` adlı bir değişkende sakla.

In [17]:
"""Tüm ticket'ları `vectorize_layer` ile `(max_len,)` tensor listesine çevir."""
all_tokenized = [vectorize_layer(ticket) for ticket in tickets]


Artık 372 elemanlı bir `list` olmalı; her eleman 56 uzunluğunda 1-boyutlu bir tensor.

In [18]:
from nbresult import ChallengeResult

result = ChallengeResult('tokenization',
    list_length = len(all_tokenized),
    contains_tensor = tf.is_tensor(all_tokenized[0]),
    tensor_shape = all_tokenized[0].shape[0]
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 3 items

test_tokenization.py::TestTokenization::test_len PASSED                  [ 33%]
test_tokenization.py::TestTokenization::test_seq_length PASSED           [ 66%]
test_tokenization.py::TestTokenization::test_type PASSED                 [100%]

============================== 3 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/tokenization.pickle

git commit -m 'Completed tokenization step'

git push origin master



Artık tüm cümlelerimiz token dizilerine dönüştüğüne göre, X ve y'nin tam olarak ne olacağını dikkatlice düşünelim. 372 cümle model eğitmek için çok fazla bilgi gibi görünmüyor — gerçekten de değil; bunu biraz da örnek amaçlı hızlı eğitim süresi olsun diye böyle tuttuk. Ama cümleleri daha küçük X-y çiftlerine bölerek veri miktarını artırabiliriz.

Modelimiz, bir cümlede bir sonraki kelimeyi tahmin etmeli; yani o ana kadar gördüğü kelimelerden başka bir şey bilmiyor. Aşağıda en bariz “sonraki kelime” eğitim örneğini görebilirsin.

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/gpt_scratch_1.png">

Ama cümleden çıkarabileceğimiz başka birçok X ve y seti de var:

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/quick_brown_examples.png">

8 kelimelik bir cümleden 7 (yani n - 1) tane X-y çifti ürettik! Bunu daha hızlı/etkin nasıl yapabiliriz? Cümleyi toplam 7 kez kopyaladığımızı düşünelim.

<img src = https://d37p7d5kaxknzw.cloudfront.net/projects/quick_brown_examples_X.png>

Kırmızıyla altı çizili kısımlar bizim ilgilendiğimiz yerler — harika olurdu eğer bir mask kullanıp istemediklerimizi yok sayabilsek. 

<img src = https://d37p7d5kaxknzw.cloudfront.net/projects/quick_brown_examples_masking.png>

İşte tam olarak `tf.linalg.band()` (ısınma egzersizinde görmüştün!) bizim için bunu yapacak!

Son olarak y'lerimizi düşünelim — tensordaki her satırdan bir tane seçebiliriz ama daha verimli bir yol var:

<img src = https://d37p7d5kaxknzw.cloudfront.net/projects/quick_brown_examples_y.png>

Yani y'lerimiz aslında sadece cümlemizin `sequence[1:]` hali olacak.

Bu örnek (dummy) tensor üzerinde aşağıdaki işlemleri yap:
1) `tf.tile()` ile orijinal dizinin tekrarlarından oluşan 55x56 şekilli bir tensor oluştur. Önce `tf.expand_dims` kullanman gerekecek; böylece `tile` işlemini iki boyutta yapabilirsin.

In [19]:
dummy_tensor = tf.range(0,56)

In [20]:
"""`(56,)` diziyi `(1,56)` genişlet; `tile` ile `(55,56)` tekrarlı batch oluştur."""
dummy_tiled = tf.tile(tf.expand_dims(dummy_tensor, 0), [55, 1])


2) `tf.linalg.band_part()` ile tensorün üst üçgenini maskele (yani yok say).


In [21]:
"""`tf.linalg.band_part(..., -1, 0)`: satır r için sütun c>r olan future pozisyonları sıfırlar (causal mask)."""
dummy_masked = tf.linalg.band_part(dummy_tiled, -1, 0)


3) Buna karşılık gelen y'leri üret.

In [22]:
"""Hedef `y`: bir adım kaydırılmış sequence (`sequence[1:]`, shape `(55,)`)."""
y = dummy_tensor[1:]


Bu adımlar çalışınca, `return X, y` döndürecek fonksiyonu doldur (burada X şekli (55,56), y şekli (55,) olmalı).

In [23]:
def X_y_creator(sequence_tensor):
    """Teacher forcing: `tile` + `band_part` ile maskeli `X`, `y` ise `next token` (offset 1)."""
    max_seq = tf.shape(sequence_tensor)[0]
    n_rows = max_seq - 1
    X_tiled = tf.tile(tf.expand_dims(sequence_tensor, 0), [n_rows, 1])
    X_s = tf.linalg.band_part(X_tiled, -1, 0)
    y_s = sequence_tensor[1:max_seq]
    return X_s, y_s


Aşağıdaki hücreyi hatasız çalıştırabilmelisin: 

In [24]:
X, y = X_y_creator(all_tokenized[0])

In [25]:
from nbresult import ChallengeResult


result = ChallengeResult('xy_creater',
    list_length = len(X_y_creator(all_tokenized[0])),
    X_shape = tuple(X_y_creator(all_tokenized[0])[0].shape),
    y_shape = tuple(X_y_creator(all_tokenized[0])[1].shape)
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 3 items

test_xy_creater.py::TestXyCreater::test_X_shape PASSED                   [ 33%]
test_xy_creater.py::TestXyCreater::test_list_length PASSED               [ 66%]
test_xy_creater.py::TestXyCreater::test_type PASSED                      [100%]

============================== 3 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/xy_creater.pickle

git commit -m 'Completed xy_creater step'

git push origin master



Şimdi bu fonksiyonu `all_tokenized` listesindeki her elemana uygula.

In [26]:
Xs = []
ys = []
for sequence in all_tokenized:
    X_i, y_i = X_y_creator(sequence)
    Xs.append(X_i)
    ys.append(y_i)


Şimdi küçük bir toparlama yapalım: Tüm bu `for` döngülerini verimsiz şekilde yazdık ama amacımız süreci adım adım anlayıp metodik ilerlemekti. `X`'ler için bir tensor listemiz ve `y`'ler için bir `y` listemiz var; `tf.concat` ile bunları birleştirip 20460 eğitim örneği (hızlı büyüdü!) oluşturacağız. Bu yeni değişkenin adı `X` olsun ve şekli `(20460, 56)` olsun. Aynı birleştirme işlemini `y` için de yapıp `(20460,)` şekilli `y` değişkenine kaydet.

In [27]:
"""Tüm sequence'lerden gelen batch'leri `tf.concat` ile tek `X`, tek `y` tensor'unda birleştir."""
X = tf.concat(Xs, axis=0)
y = tf.concat(ys, axis=0)


Henüz düşünmediğimiz bir şey var! `y` değerlerimizin çoğu padding yüzünden sadece 0. Hızlıca, `y`'lerin __0 olmadığı__ yerleri bulmak için boolean bir mask oluşturalım; sonra hem X hem de y'den sadece bu örnekleri tutalım.

In [28]:
"""Padding token id (`0`) hedeflerini çıkar; `boolean_mask` ile geçerli `next-token` örnekleri tut."""
mask = y != 0
X = tf.boolean_mask(X, mask)
y = tf.boolean_mask(y, mask)


Oh be! Neredeyse 5000 tane işe yaramaz eğitim örneğini attık. Doğru şekillere ve değerlere ulaştığından emin olmak için aşağıdaki testlerle X ve y'yi kontrol et.

In [29]:
from nbresult import ChallengeResult

result = ChallengeResult('final_shapes',
    X_shape = tuple(X.shape),
    X_value = int(X[500][7]),
    y_shape = tuple(y.shape),
    y_value = int(y[356]),
    zeroes = int(tf.math.reduce_sum(tf.cast(y==0, "int32")))
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 5 items

test_final_shapes.py::TestFinalShapes::test_X_shape PASSED               [ 20%]
test_final_shapes.py::TestFinalShapes::test_sample_X_values PASSED       [ 40%]
test_final_shapes.py::TestFinalShapes::test_y_shape PASSED               [ 60%]
test_final_shapes.py::TestFinalShapes::test_y_value PASSED               [ 80%]
test_final_shapes.py::TestFinalShapes::test_zeroes PASSED                [100%]

============================== 5 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/final_shapes.pickle

git commit -m 'Completed final_shapes step'

git push origin master



Hepsi tamam! Çok iş vardı — ve bunu sadece bir kere yapman gerekecek — ama modele neyin girdiğini ve girdiden neyin tahmin edildiğini anlaman kritik.

## Modelleme

Şimdi işin zor kısmına geldik: modeli inşa etmek! Dersten hatırlayacağın gibi GPT tarzı modeller “sadece decoder” (decoder-only) olarak adlandırılır. Decoder-only GPT modelleri, Transformer mimarisinin özellikle decoder bileşenini kullanarak giriş dizilerinden çıktı dizileri üretir. Aşağıdaki diyagram GPT-2 mimarisini gösteriyor:

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/GPT2.png" width="400px">

Diyagramın sol tarafına odaklanırsan, kelimelerimizin (token'ların) yolculuğunu kolayca gözünde canlandırabilirsin.

### Adım 1. Konumsal (Positional) Encoding ve Word Embedding'ler: 

Modelin girdisi, az önce hazırladığımız token'lar. Şimdi bu girdi token'larına iki şey yapmamız gerekiyor:  

1) Normal token embedding (dünkü NLP görevlerinde yaptığımız gibi) ve ayrıca positional embedding eklemek. Hatırlatma: token embedding, bir token'ın anlamını seçtiğimiz `embedding_dimensions` sayısı boyunca bir vektöre gömmek demek.

2) Positional encoding ile her kelimenin cümlede nerede olduğuna dair ipuçlarını modele vermek; bu positional encoding, giriş embedding'lerine basitçe eklenir.

Böylece model, dizideki token'ların göreli konumlarını __ve__ her kelimenin ne “anlama geldiğini” __birlikte__ anlayabilir. Neyse ki `keras_nlp` kütüphanesindeki `TokenAndPositionEmbedding()` katmanı ile ikisini birden tek seferde yapabiliriz! Dersteki adımı hatırlamak için aşağıdaki diyagrama bak: 

<img src =https://d37p7d5kaxknzw.cloudfront.net/projects/positional_encoding_sketch.png width=300px>

### Adım 2. Transformer Bloğu: 

Gömülmüş (embedded) vektörlerimiz şimdi Transformer bloğuna giriyor; bu kısım __GPT mimarisinin kalbidir__. Buradaki attention mekanizması, modelin tahmin yaparken girdi dizisindeki farklı konumlara “dikkat” etmesini sağlar. Model, attention ağırlıkları hesaplayarak her bir giriş token'ının (artık embedding'leriyle temsil ediliyor) diğer token'larla ilişkili önemini öğrenir.

Bu süreç birkaç adımdan oluşur: önce embedded vektörlerimizi Query, Key ve Value vektörlerine projekte ederiz. Multi-head attention için bu biraz daha karmaşık olabilir (istersen [buraya](https://towardsdatascience.com/transformers-explained-visually-part-3-multi-head-attention-deep-dive-1c1ff1024853) bakabilirsin); ama bu matrisler elimizdeyken scaled dot-product attention'ı şu formülü takip ederek yaparız!

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/key-query-value.png">

Bunu yaptıktan ve embedding'leri güncelledikten sonra, Transformer bloğunun son katmanlarından geçiririz: biraz Dropout ve LayerNormalization eklenir.

Sonuç olarak Transformer bloğunun diğer tarafında güncellenmiş vektörler elde ederiz — her vektör hâlâ 512 uzunluğundadır ama artık görev açısından (bizim için “bir sonraki kelimeyi tahmin etmek”) önemleri hakkında daha fazla bilgi taşırlar.

### Adım 3. Tahmin yapmak: 

Şimdi çıktımızın ne olacağını çok dikkatli düşünmeliyiz.

Hatırla: `X` cümlenin bir noktaya kadarki kısmı, `y` ise bir sonraki kelime. Bu, tahmin açısından ne anlama geliyor? Temelde karşımızda dev bir sınıflandırma problemi var. Bir sonraki kelimeyi doğru seçmemiz gerekiyor; peki kaç seçeneğimiz var?

Cevap: vocabulary'mizde kaç kelime varsa o kadar! Çıktımız, vocabulary boyutumuz kadar nöronu olan bir Dense katman olacak. Aktivasyon fonksiyonu “softmax” olacak — yani tüm nöronlar üzerinde toplamı 1 eden olasılıklar tahmin edecek. Doğru “bir sonraki kelime”nin olasılığını mümkün olduğunca 1'e yakın yapmak istiyoruz.

Süreci aşağıda görebilirsin:

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/prediction_png_flow.png">

Peki bu bizim için kod tarafında nasıl görünüyor? Aşağıda tanımladığımız model fonksiyonu var; içinde birkaç boşluk bırakıldı:
    

In [30]:
from keras_nlp.layers import TokenAndPositionEmbedding
from keras import Input, layers

def create_model(max_sequence_length, vocab_size, embedding_dimension):

    # 1. First up we define a layer that just grabs the inputs to our model
    # We use the standard Input() layer and we know that each X going into
    # our model is going to be 56 long!
    inputs = Input(shape=(max_len,), dtype=tf.int32)

    # 2. Next we give our tokens Positional and Regular Embeddings which is done
    # by a nicely built layer that takes these arguments!
    x = TokenAndPositionEmbedding(vocab_size,
                                  max_sequence_length,
                                  embedding_dimension,
                                  mask_zero = True)(inputs)

    # 3. This part we are going to define in a moment - don't worry
    # about it for now - we'll come back to it!
    x = TransformerBlock(num_heads=4,
                         embed_dim=embedding_dimension,
                         ff_dim=embedding_dimension * 4)(x)


    # 4. This is just a regular Dropout layer that you've
    # seen earlier in the week that helps our model avoid overfitting
    x = layers.Dropout(0.4)(x)


    # 5. At this point in the model we'll have tensors with
    # shape (batch_size, sequence_length, embedding_dimension) but
    # we want to squish it down so we use GlobalAveragePooling1d. All
    # this does is average elements across our sequence and
    # squish them into a (batch_size, embedding_dimension) tensor.
    x = layers.GlobalAveragePooling1D()(x)


    # 6. Finally we just need to have our "classification" layer
    # which needs to be as large as our vocabulary size
    outputs = layers.Dense(vocab_size, activation='softmax')(x)


    # Now we just just stick our model together with the Functional API
    # and compile with "adam" for our optimizer and
    # "sparse_categorical_crossentropy" to compute the loss
    # between our predicted labels and our actual labels.
    # We'll talk about perplexity a little later.

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer="adam",
        loss='sparse_categorical_crossentropy',
        metrics=[keras_nlp.metrics.Perplexity(), 'accuracy']
    )
    return model

In [46]:
create_model(50, 50, 50)

/home/tumay/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block_2' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


<Functional name=functional_5, built=True>

Yukarıdaki hücreyi çalıştırırsan hata alacaksın! Neden? Çünkü birçok katmanı henüz tanımlamadık ve TransformerBlock'u hâlâ implement etmemiz gerekiyor — asıl sihir burada oluyor!

Transformer bloğunu kodlayabilmek için önce attention mekanizmasını kodlamamız gerekiyor. Aşağıda bunun neye benzediğine dair bir hatırlatma var.

<img src = "https://d37p7d5kaxknzw.cloudfront.net/projects/key-query-value.png">

Bu katmanın yaptığı şey; embedded girdilerimizi üç matrise projekte etmek — query, key ve value — ve bu üçlü arasındaki etkileşimle kelimeler için daha iyi embedding'ler üretmek. Adım adım parçalayalım:

1. Q matrisi ile K matrisinin transpozunu çarparız.
2. Bunu model boyutunun kareköküne böleriz.
3. Son boyutta Softmax alarak ölçeklenmiş skorları (scaled scores) elde ederiz.
4. Bu softmax skorlarını V matrisi ile çarparız.

In [33]:
def coded_attention(query, key, value):
        """Scaled dot-product attention; bu notebook ölçeği `sqrt(512)≈22.6` (`tf.float32`)."""
        # Step 1: Matrix multiply the query with the transpose of the key
        scores = tf.matmul(query, key, transpose_b=True)

        # Step 2: Divide this matrix by the square root of the hidden dimension
        # In our case this dimension will be 512 (with the square root being 22.6).
        # You will have to use tf.cast(22.6, tf.float32) so that the two matrices can interact

        scaled_scores = scores / tf.cast(22.6, tf.float32)

        # Step 3:
        # Compute the softmax_scores - use tf.nn.softmax(scaled_score, axis = ?)
        # Think about what dimension we should be using our softmax along -
        # it'll need to be our last dimension!
        softmax_scores = tf.nn.softmax(scaled_scores, axis=-1)

        # Step 4:
        # Matrix multiply the weights matrix with the value matrix
        # and set this to be your "output"
        output = tf.matmul(softmax_scores, value)


        # Return *both* the output and the softmax_scores as a tuple
        return output, softmax_scores


Aşağıdaki hücreyi çalıştırarak fonksiyonunu bazı örnek (dummy) tensor'lerle test et:
    

In [34]:
# Dummy tensor for query
example_query = tf.constant([[0.1, 0.2],
                     [0.3, 0.4]])

# Dummy tensor for key
example_key = tf.constant([[0.5, 0.6],
                   [0.7, 0.8]])

# Dummy tensor for value
example_value = tf.constant([[0.9, 1.0],
                     [1.1, 1.2]])
# Test your function
coded_attention(example_query, example_key, example_value)

(<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[1.0001327, 1.1001327],
        [1.0003097, 1.1003097]], dtype=float32)>,
 <tf.Tensor: shape=(2, 2), dtype=float32, numpy=
 array([[0.49933627, 0.5006637 ],
        [0.49845132, 0.50154865]], dtype=float32)>)

In [35]:
from nbresult import ChallengeResult


example_query = tf.constant([[0.1, 0.2],
                     [0.3, 0.4]])


example_key = tf.constant([[0.5, 0.6],
                   [0.7, 0.8]])


example_value = tf.constant([[0.9, 1.0],
                     [1.1, 1.2]])
output = coded_attention(example_query, example_key, example_value)

result = ChallengeResult('attention',
    len_output = len(output),
    output_shape = tuple(output[0].shape),
    output_value = tuple(output[0][-1].numpy())
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 3 items

test_attention.py::TestAttention::test_len PASSED                        [ 33%]
test_attention.py::TestAttention::test_output_shape PASSED               [ 66%]
test_attention.py::TestAttention::test_output_value PASSED               [100%]

============================== 3 passed in 0.10s ===============================


💯 You can commit your code:

git add tests/attention.pickle

git commit -m 'Completed attention step'

git push origin master



Bunu çalıştırdıktan sonra, el ile kodladığımız attention'ı daha büyük MultiHeadAttention'a ve daha da büyük TransformerBlock'a gömebiliriz.

__Eğer aşağıdaki bloğun her adımını tek tek anlamak istiyorsan, bunu sonra detaylıca yap__; şimdilik mimariyi tanımlamak için aşağıdaki hücreyi çalıştırıp devam edebilirsin — neredeyse geldik!

In [37]:
from keras import layers, Model, Sequential
import keras_nlp

class MultiHeadAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.projection_dim = embed_dim // num_heads
        self.query_dense = layers.Dense(embed_dim)
        self.key_dense = layers.Dense(embed_dim)
        self.value_dense = layers.Dense(embed_dim)
        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, query, key, value):
        # Your coded attention function goes here
        return coded_attention(query, key, value)

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, 56, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        # Here, we "project" into long query, key, value vectors
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)
        batch_size = tf.shape(query)[0]

        # We rearrange the projections for each head
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)

        # We perform attention on our QKV for our heads
        attention, weights = self.attention(query, key, value)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim))
        output = self.combine_heads(concat_attention)
        return output

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.ffn = Sequential(
            [layers.Dense(ff_dim, activation="relu"),
             layers.Dense(embed_dim)]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        attention_output = self.attention(inputs)
        attention_output = self.dropout1(attention_output)
        out1 = self.layernorm1(inputs + attention_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)


Şimdi tek yapmamız gereken her şeyi birleştirmek. Aşağıdaki fonksiyon bizim için hepsini “dikişleyip” bir araya getirecek! Aslında sadece 5 katman var (ama az önce gördüğümüz gibi, bunlardan biri oldukça karmaşıktı!)

In [38]:
def create_model(max_sequence_length, vocab_size, embedding_dimension):

    inputs = Input(shape=(max_len,), dtype=tf.int32)
    x = TokenAndPositionEmbedding(vocab_size,
                                  max_sequence_length,
                                  embedding_dimension,
                                  mask_zero = True)(inputs)
    x = TransformerBlock(num_heads=4,
                         embed_dim=embedding_dimension,
                         ff_dim=embedding_dimension)(x)
    x = layers.Dropout(0.4)(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(vocab_size, activation='softmax')(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer="adam",
        loss='sparse_categorical_crossentropy',
        metrics=[keras_nlp.metrics.Perplexity(), 'accuracy']
    )
    return model

Modelimizi şu değerlerle oluşturacağız:
- `max_sequence_length` = En uzun sequence uzunluğumuz
- `vocab_size` = Vocabulary'deki benzersiz kelime sayısı
- `embedding_dimension` = 512 (bu boyut, bu veri büyüklüğü için genelde iyi çalışır)

In [39]:
"""`max_sequence_length=max_len`, `vocab_size`, `embedding_dimension=512` ile model kur."""
model = create_model(56, vocab_size, 512)


/home/tumay/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model özetine (summary) göz at. 

In [40]:
"""Katman şekilleri ve parametre sayısı için `summary`."""
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 56)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_3  │ (None, 56, 512)        │       617,472 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 56, 512)        │     1,577,984 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 56, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 1150)           │       589,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,785,406 (10.63 MB)

 Trainable params: 2,785,406 (10.63 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
from nbresult import ChallengeResult

model = create_model(56, 1150, 512)

result = ChallengeResult('model',
    params = model.count_params()
)

result.write()
print(result.check())

/home/tumay/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block_1' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(



============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/tumay/.pyenv/versions/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/tumay/code/data-science-gpt-from-zero/tests
plugins: anyio-4.8.0, dash-3.3.0, typeguard-4.4.2
collecting ... collected 1 item

test_model.py::TestModel::test_params PASSED                             [100%]

============================== 1 passed in 0.01s ===============================


💯 You can commit your code:

git add tests/model.pickle

git commit -m 'Completed model step'

git push origin master



Geriye kalan tek şey, modelimizi `X` ve `y` ile fit etmek. Batch size 32 iyi olur; epoch sayısı olarak da 20 deneyebilirsin. Bu sırada aşağıdaki perplexity açıklamasını okuyup çayını yudumlayabilirsin; modelimiz işini yapsın! 

In [42]:
# Before fitting, we need to expand the dims of y to make it work
# with the perplexity measure when using the latest version of tf.
# This changes the shape from (n_samples, ) to (n_samples, 1).
y = tf.expand_dims(y, axis=1)

In [43]:
"""`sparse_categorical_crossentropy` ile `fit`; batch 32, epoch 20 (uzun sürebilir)."""
history = model.fit(X, y, batch_size=32, epochs=20)


Epoch 1/20


/home/tumay/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'position_embedding' (of type PositionEmbedding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/tumay/.pyenv/versions/workintech/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'multi_head_attention_1' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


533/533 ━━━━━━━━━━━━━━━━━━━━ 44s 78ms/step - accuracy: 0.1313 - loss: 4.7110 - perplexity: 111.1581
Epoch 2/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 81s 76ms/step - accuracy: 0.4469 - loss: 2.5205 - perplexity: 12.4346
Epoch 3/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.6225 - loss: 1.6096 - perplexity: 5.0009
Epoch 4/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 39s 73ms/step - accuracy: 0.7177 - loss: 1.1175 - perplexity: 3.0572
Epoch 5/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 41s 73ms/step - accuracy: 0.7810 - loss: 0.8395 - perplexity: 2.3151
Epoch 6/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 40s 75ms/step - accuracy: 0.8186 - loss: 0.6505 - perplexity: 1.9165
Epoch 7/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 40s 75ms/step - accuracy: 0.8437 - loss: 0.5561 - perplexity: 1.7438
Epoch 8/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 40s 75ms/step - accuracy: 0.8632 - loss: 0.4733 - perplexity: 1.6053
Epoch 9/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 40s 72ms/step - accuracy: 0.8777 - loss: 0.4176 - perplexity: 1.5183
Epoch 10/20
533/533 ━━━━━━━

<details> 
    <summary> Perplexity Açıklaması için TIKLA</summary>
<br>
Perplexity, dil modelleri (Language Models) bağlamında, modelin bir kelime dizisindeki bir sonraki kelimeyi/token'ı ne kadar iyi tahmin ettiğini ölçmek için sık kullanılan bir metriktir. Modelin belirli bir kelime dizisine ne kadar yüksek olasılık atadığını değerlendirir.

Matematiksel olarak perplexity, cross-entropy kavramı ile hesaplanır. Perplexity skoru, bir veri setindeki kelime başına ortalama cross-entropy'nin üstelidir. Formül şöyledir:

$$
\text{Perplexity} = \exp\left(-\frac{{\sum_{{i=1}}^{{N}} \log(p(w_i))}}{{N}}\right)
$$

Burada $N$, veri setindeki toplam kelime sayısını; $(p(w_i))$ ise modelin dizideki $(i)$'inci kelimeye atadığı olasılığı temsil eder.

Formül; her kelime için modelin tahmin ettiği olasılıkların logaritmasını alıp toplamayı içerir. Bu toplamı toplam kelime sayısına $(N)$ bölüp üstelini aldığımızda perplexity skorunu elde ederiz.

Daha düşük perplexity, modelin tahminlerinde daha “emin” ve daha doğru olduğunu (doğru kelimelere daha yüksek olasılık verdiğini) gösterir. Daha yüksek perplexity ise modelin daha belirsiz ve daha az doğru tahmin yaptığını ima eder.

Perplexity, farklı dil modellerini karşılaştırmak veya eğitim sırasında modelin ilerleyişini takip etmek için de kullanılır. Genelde daha düşük perplexity daha iyidir.</details>


### Modelimizi kullanmak

Hatırlayacağın gibi GPT tarzı modeller basitçe şöyle çalışır: Bir giriş dizisi alır, bir sonraki kelimeyi tahmin eder, o kelimeyi mevcut girişe ekler ve tekrar tekrar tahmin etmeye devam eder!

Şimdi `starter_string` girdisini alan bir `generate` fonksiyonu yazmak istiyoruz.

Sonra, ``range(max_len - len(starter_string.split()))`` kadar dönen bir `for` döngüsü içinde:

1. `vectorize_layer` ile bunu bir `token_tensor`'a çevir
2. `model.predict(token_tensor)` ile modelden vocab_size boyutunda bir çıktı al (model girdisi doğru şekle gelsin diye `tf.expand_dims(token_tensor, 0)` kullanman gerekecek!)
3. `tf.argmax()` ile en olası (en büyük değerli) kelimenin indeksini bul (bu aslında her adımda en olası kelimeyi seçtiğimiz için “greedy” bir yaklaşımdır)
4. Daha önce oluşturduğumuz `index_lookup` sözlüğü ile bu indeksi tekrar kelimeye çevir
5. Bu kelimeyi giriş string'ine ekle ve tekrarla — şu iki durumda durdur/bitir:

İki önemli nokta: 

1) Eğer "eos" kelimesini tahmin edersek, model cümlenin bittiğini söylüyor demektir; bu yüzden `starter_string`'i mevcut haliyle `return` et.

2) String'in uzunluğu (boşluklardan split edince) 55 olursa, yine `starter_string`'i mevcut haliyle `return` et.

Not: Kelimeyi diziye eklerken araya boşluk koymayı unutma!

In [44]:
def generate(starter_string):
    """Greedy decoding: `argmax` ile sonraki kelimeyi seç; `eos` veya max uzunlukta dur."""
    current = starter_string
    for _ in range(max_len - len(starter_string.split())):
        token_tensor = tf.expand_dims(vectorize_layer(current), 0)
        pred = model.predict(token_tensor, verbose=0)
        next_id = int(tf.argmax(pred, axis=-1).numpy()[0])
        next_word = index_lookup[next_id]
        if next_word == "eos":
            return current
        current = current + " " + next_word
        if len(current.split()) >= 55:
            return current
    return current


In [45]:
generate("I need")

'I need on a detect my scikit-learn to moving for network traffic or cybersecurity data. how can i leverage techniques like autoencoders, one-class svm, or using specialized algorithms like isolation forest or adasyn, or adasyn, or adasyn, or leveraging deep approaches to effectively model and address class imbalance in spatial data? data? data?'

Tebrikler!!! Kendi GPT modelini ilk prensiplerden başlayarak inşa ettin 💪

Doğal olarak, yaptıklarımızı yapmanın __çok__ daha verimli yolları var — neredeyse hiçbir zaman kendi Transformer bloğunu, attention mekanizmanı ya da komple bir GPT modelini sıfırdan yazmayacaksın. HuggingFace, zor kodlamanın büyük kısmını bizden soyutluyor; bu yüzden mevcut modelleri fine-tune etmek çoğu zaman çok daha etkili.

Üzerinde çalıştığımız dataset oldukça küçük ve tokenization da çok basit (şu an her kelimeye ayrı bir token veriliyor ve noktalama işaretleri de içeride kalıyor). Daha “kirli” verilerle çalışmak istersen, kodunu Python dosyalarına taşıyıp düzenleyebilir; sonra da buradaki adımları şu 10.000 StackOverflow cevabını içeren gerçek veri üzerinde tekrar uygulayabilirsin: [burada](https://d37p7d5kaxknzw.cloudfront.net/projects/answers.csv). Böylece daha karmaşık bir generator da inşa edebilirsin.